# Olist: локальный baseline риска опоздания

Готовая историческая таблица реальных заказов поставляется в `data/olist_orders.csv.gz`.
Сеть, Kaggle, PostgreSQL и MLflow для этого ноутбука не нужны. Источник — Olist
`olistbr/brazilian-ecommerce`; данные и производная таблица: **CC BY-NC-SA 4.0**.
См. `data/README.md`, `data/LICENSE.md`, `data/metadata.json`.

Модель оценивает риск при покупке. Фактическая доставка используется **только** для
целевой метки `late`, а отзывы/даты передачи перевозчику не входят в признаки.
Когорта ограничена доставленными заказами: это учебный baseline, не production-оценка
всех заказов. Обещанный срок `est_days` известен в момент покупки.

В ДЗ нужно реализовать четыре задачи pipeline: extract → validate → prepare → train.
Агрегация и алгоритм модели уже предоставлены; переписывать нейросеть не требуется.

In [ ]:
import hashlib
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

candidates = [Path.cwd(), Path.cwd() / "starter", Path.cwd().parent]
ROOT = next((path for path in candidates if (path / "src/olist_model.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Запустите ноутбук из starter/ (или корня проекта)")
sys.path.insert(0, str(ROOT / "src"))
from olist_model import FEATURE_COLUMNS, split_by_time, fit_model, save_predictor, load_predictor

DATASET = Path(os.environ.get("OLIST_DATASET", str(ROOT / "data/olist_orders.csv.gz")))
OUTPUT = Path(os.environ.get("OLIST_NOTEBOOK_OUTPUT", str(Path(os.environ.get("OLIST_WORKDIR", "outputs")) / "notebook")))
OUTPUT.mkdir(parents=True, exist_ok=True)
CUTOFF = os.environ.get("OLIST_CUTOFF", "2017-07-31")
EPOCHS = int(os.environ.get("OLIST_EPOCHS", "5"))
SEED = int(os.environ.get("OLIST_SEED", "42"))

In [ ]:
metadata = json.loads((DATASET.parent / "metadata.json").read_text(encoding="utf-8"))
actual_sha256 = hashlib.sha256(DATASET.read_bytes()).hexdigest()
assert actual_sha256 == metadata["sha256"], "Dataset checksum mismatch"
df = pd.read_csv(DATASET)
assert list(df.columns) == metadata["columns"]
assert len(df) == metadata["rows"] and df.order_id.is_unique
print(json.dumps({"rows": len(df), "sha256": actual_sha256,
                  "date_min": metadata["date_min"], "date_max": metadata["date_max"],
                  "features": FEATURE_COLUMNS}, indent=2))

## Разбиение по времени и train-only preprocessing

Train: покупка **до** `2017-07-31 00:00:00`; validation: начиная с этой даты.
В обоих наборах необходимы обе метки 0/1. Никакого случайного разбиения.
`fit_model` учит масштабирование и словари категорий **только на train**;
новые категории validation/inference дают нулевой one-hot блок. Числовые
NaN/Inf отвергаются; пропуски категорий обрабатываются явно.

Это небольшой CPU MLP, class-weighted BCE. `validation_pr_auc` означает
`average_precision_score` (ступенчатая PR-площадь), не трапецеидальную площадь.
Из-за взвешенного loss выходы — некалиброванные положительные вероятностные scores.
Сравниваем с долей положительного класса; не подбираем параметры на validation.

In [ ]:
train_df, valid_df = split_by_time(df, cutoff=CUTOFF)
print(json.dumps({"train_rows": len(train_df), "validation_rows": len(valid_df),
                  "train_late_rate": float(train_df.late.mean()),
                  "validation_prevalence_baseline": float(valid_df.late.mean())}, indent=2))
predictor, history = fit_model(train_df, valid_df, epochs=EPOCHS, seed=SEED)
for row in history:
    print(json.dumps(row, allow_nan=False))

## Полный артефакт и безопасное восстановление

Сохраняем `weights.pt` (только state_dict тензоров) и `preprocessing.json`
(числовые параметры и словари train). `load_predictor` создаёт известную
архитектуру и читает веса с `weights_only=True`, а не десериализует Python-модель.
Не загружайте произвольные недоверенные артефакты: это не sandbox от исчерпания ресурсов.
Проверяем предсказания после восстановления; артефакты пишутся в
`$OLIST_WORKDIR/notebook` (по умолчанию `outputs/notebook`), не в read-only src/data.

In [ ]:
before = predictor.predict(valid_df.drop(columns="late"))
assert np.isfinite(before).all() and ((0 <= before) & (before <= 1)).all()
save_predictor(predictor, OUTPUT / "model")
restored = load_predictor(OUTPUT / "model")
after = restored.predict(valid_df.drop(columns="late"))
np.testing.assert_array_equal(before, after)
summary = {
    "dataset_sha256": actual_sha256, "cutoff": CUTOFF, "epochs": EPOCHS, "seed": SEED,
    "train_rows": len(train_df), "validation_rows": len(valid_df),
    "validation_prevalence_baseline": float(valid_df.late.mean()),
    "validation_pr_auc": history[-1]["validation_pr_auc"], "history": history,
    "safe_reload_predictions_equal": bool(np.array_equal(before, after)),
}
(OUTPUT / "metrics.json").write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2, allow_nan=False))
print(json.dumps({"sample_probabilities": after[:5].tolist()}, allow_nan=False))